In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import pickle as pkl

## Models So Far

### Readmission classifiers — predicts: will this patient be readmitted within 30 days? (binary)
Trained on: demographics + vitals + labs (full `df_encoded_v3` feature set)

| Model | Recall (readmit) | Precision | F1 | AUC-ROC | Accuracy |
|---|---|---|---|---|---|
| **`model_readmit` (undersampled + scale_pos_weight)** ⭐ best recall | **0.66** | 0.44 | 0.53 | 0.57 | 0.53 |
| `model_readmit` (baseline, no resampling) | 0.34 | 0.48 | 0.40 | 0.60 | 0.58 |

Undersampling + scale_pos_weight roughly doubles recall vs. baseline, at the cost of precision/accuracy — worth it since catching readmits matters more than false alarms here.

### LOS (length of stay) regressors — predicts: how many days will this ICU stay last? (continuous, log-target)

| Model | Features | MAE (days) | RMSE (days) | R² (ln scale) |
|---|---|---|---|---|
| **`model_v3`** ⭐ best MAE | demographics + vitals + labs | **2.10** | 4.99 | 0.196 |
| `model_v2` | demographics + vitals | 2.12 | 5.01 | 0.189 |
| `model` (log-target) | demographics only | 2.36 | 5.36 | 0.195 |
| `model` (raw-target, earliest) | demographics only | — | 5.22 | 0.043 |

Baselines for comparison: guessing the mean LOS = 3.01 days MAE; guessing the median = 2.57 days MAE. `model_v3` beats both.

## DataFrames So Far

### Raw MIMIC-IV tables
| Name | Purpose | Shape |
|---|---|---|
| `admissions` | Raw hospital admission records | (546,028, —) |
| `patients` | Raw patient demographics | (364,627, —) |
| `icustays` | Raw ICU stay records, one row per stay | (94,458, —) |
| `diagnoses` | Raw ICD diagnosis codes, many rows per admission | (~6M, —) |

### Derived/aggregated tables
| Name | Purpose | Shape |
|---|---|---|
| `dx_count` | Diagnosis count per admission (comorbidity proxy) | (per hadm_id) |
| `vitals_wide` | First-24hr vitals (HR, BP, SpO2, GCS, etc.), aggregated per stay | (per stay_id) |
| `labs_wide` | First-24hr labs (creatinine, WBC, glucose, etc.), aggregated per stay | (per stay_id) |

### Main feature-building chain
| Name | Purpose | Shape |
|---|---|---|
| `df` | Base merge: icustays + admissions + patients + diagnosis count + readmit label | (94,458, 31) |
| `df_encoded` | `df` + one-hot encoded categoricals + temporal features | (94,458, 74) |
| `df_encoded_v2` | `df_encoded` + vitals merged in | (94,458, 103) |
| **`df_encoded_v3`** | `df_encoded_v2` + labs merged in — **final full feature table** | (94,458, 134) |
| `model_df` | Final LOS model matrix (IDs/leakage cols dropped) | (94,444, 113) |
| `readmit_model_df` | Final readmission model matrix (IDs/leakage/`los` dropped) | (43,488, 114) |

In [ ]:
MIMIC_ROOT = 'E:/MedAgent/data/raw/physionet.org/files/mimiciv/3.1'

admissions = pd.read_csv(f'{MIMIC_ROOT}/hosp/admissions.csv.gz')
patients = pd.read_csv(f'{MIMIC_ROOT}/hosp/patients.csv.gz')
icustays = pd.read_csv(f'{MIMIC_ROOT}/icu/icustays.csv.gz')

chartevents_path = f'{MIMIC_ROOT}/icu/chartevents.csv.gz'
labevents_path = f'{MIMIC_ROOT}/hosp/labevents.csv.gz'

In [ ]:
print("admissions:", admissions.shape)
print("patients:", patients.shape)
print("icustays:", icustays.shape)

In [ ]:
df = icustays.merge(admissions, on=['subject_id', 'hadm_id'], how='inner')
df = df.merge(patients, on='subject_id', how='inner')

df.shape


In [ ]:
df[['admittime', 'gender', 'race']].head(5)

In [ ]:
df['race'].unique()

In [ ]:
def simplify_race(race):
    race = race.upper()

    if 'WHITE' in race:
        return 'White'
    elif 'BLACK' in race:
        return 'Black'
    elif 'ASIAN' in race:
        return 'Asian'
    elif 'HISPANIC' in race or 'LATINO' in race:
        return 'Hispanic/Latino'
    elif 'SOUTH AMERICAN' in race:
        return 'Hispanic/Latino'
    elif 'AMERICAN INDIAN' in race or 'ALASKA NATIVE' in race:
        return 'American Indian/Alaska Native'
    elif 'NATIVE HAWAIIAN' in race or 'PACIFIC ISLANDER' in race:
        return 'Native Hawaiian/Pacific Islander'
    elif 'PORTUGUESE' in race:
        return 'White'  # MIMIC groups Portuguese under white ancestry
    elif 'MULTIPLE RACE' in race:
        return 'Multiple Race/Ethnicity'
    elif race in ('UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER'):
        return 'Unknown/Declined'
    else:
        return 'Other'


df['race_simplified'] = df['race'].apply(simplify_race)
df['race_simplified']


In [ ]:
"""

/// chartevents_path = f'{MIMIC_ROOT}/icu/chartevents.csv.gz'
labevents_path = f'{MIMIC_ROOT}/hosp/labevents.csv.gz'


simpler is better?
"""

In [ ]:
for column in df.columns:
    print(f"{column}: {df[column].iloc[0]}")

## Timestamp Columns — What They Mean

- **`admittime`** — when the patient was admitted to the hospital overall (2180-07-23, 12:35 PM)
- **`intime`** — when the patient was moved into the ICU specifically (2180-07-23, 2:00 PM — about 1.5 hours after hospital admission)
- **`outtime`** — when the patient was moved out of the ICU (2180-07-23, 11:50 PM — so they spent about 10 hours in the ICU)
- **`dischtime`** — when the patient was discharged from the hospital entirely (2180-07-25, 5:55 PM — about 2 days after leaving the ICU)



In [ ]:
col = ['admission_type', 'admission_location', 'insurance', 'marital_status', 'race_simplified', 'first_careunit']

df = pd.get_dummies(df, columns= col, drop_first=True)

df.head()

In [ ]:
df

In [ ]:
MIMIC_ROOT = 'E:/MedAgent/data/raw/physionet.org/files/mimiciv/3.1'  # adjust if your path differs

diagnoses = pd.read_csv(f'{MIMIC_ROOT}/hosp/diagnoses_icd.csv.gz')

# only needed if `admissions` isn't already loaded either
admissions = pd.read_csv(f'{MIMIC_ROOT}/hosp/admissions.csv.gz')

In [ ]:


# Ensure timestamp columns are actual datetimes (not strings) before extracting from them
df['admittime'] = pd.to_datetime(df['admittime'])
df['edregtime'] = pd.to_datetime(df['edregtime'])
df['edouttime'] = pd.to_datetime(df['edouttime'])

# 1. gender_binary — 1 = Male, 0 = Female
df['gender_binary'] = (df['gender'] == 'M').astype(int)

# 2. age_at_admission — anchor_age adjusted for years passed since anchor_year
df['age_at_admission'] = df['anchor_age'] + (df['admittime'].dt.year - df['anchor_year'])

# 3. admit_hour — hour of day (0-23) the patient was admitted
df['admit_hour'] = df['admittime'].dt.hour

# 4. admit_dayofweek — day of week (0=Monday ... 6=Sunday) of admission
df['admit_dayofweek'] = df['admittime'].dt.dayofweek

# 5. ed_duration_hours — how long the patient spent in the ED before ICU admission
df['ed_duration_hours'] = (df['edouttime'] - df['edregtime']).dt.total_seconds() / 3600

# 6. num_diagnoses — comorbidity count, merged in from the diagnoses table
dx_count = diagnoses.groupby('hadm_id').size().reset_index(name='num_diagnoses')
df = df.drop(columns=['num_diagnoses'], errors='ignore')  # avoid duplicate cols if re-running
df = df.merge(dx_count, on='hadm_id', how='left')
df['num_diagnoses'] = df['num_diagnoses'].fillna(0).astype(int)

# 7. readmit_30d — 1 if patient readmitted within 30 days, 0 if not, NaN if no next admission exists
admissions_sorted = admissions.sort_values(['subject_id', 'admittime']).copy()
admissions_sorted['admittime'] = pd.to_datetime(admissions_sorted['admittime'])
admissions_sorted['dischtime'] = pd.to_datetime(admissions_sorted['dischtime'])
admissions_sorted['next_admittime'] = admissions_sorted.groupby('subject_id')['admittime'].shift(-1)
admissions_sorted['days_to_next_admit'] = (admissions_sorted['next_admittime'] - admissions_sorted['dischtime']).dt.total_seconds() / 86400

admissions_sorted['readmit_30d'] = (
    (admissions_sorted['days_to_next_admit'] >= 0) &
    (admissions_sorted['days_to_next_admit'] <= 30)
).astype('Int64')
admissions_sorted.loc[admissions_sorted['days_to_next_admit'].isna(), 'readmit_30d'] = pd.NA

df = df.drop(columns=['readmit_30d'], errors='ignore')  # avoid duplicate cols if re-running
df = df.merge(admissions_sorted[['hadm_id', 'readmit_30d']], on='hadm_id', how='left')

print("df shape:", df.shape)
print("New columns present:", all(c in df.columns for c in 
      ['gender_binary', 'age_at_admission', 'admit_hour', 'admit_dayofweek', 
       'ed_duration_hours', 'num_diagnoses', 'readmit_30d']))

In [ ]:
df

In [ ]:
df.columns

In [ ]:
for column in df.columns:
    print(f"{column}: {df[column].iloc[0]}")

In [ ]:
redundant_cols = [
    'race',              # replaced by race_simplified_* one-hot columns
    'gender',            # replaced by gender_binary
    'anchor_age',        # used to compute age_at_admission
    'anchor_year',       # used to compute age_at_admission
    'edregtime',         # used to compute ed_duration_hours
    'edouttime',         # used to compute ed_duration_hours
    'admittime',         # used to compute age_at_admission, admit_hour, admit_dayofweek
]

df = df.drop(columns=redundant_cols)

print("df shape after dropping redundant columns:", df.shape)

In [ ]:
df.columns

In [ ]:
leakage_and_id_cols = [
    'subject_id', 'hadm_id', 'stay_id', 'admit_provider_id',
    'last_careunit', 'intime', 'outtime', 'dischtime', 'deathtime',
    'discharge_location', 'hospital_expire_flag', 'dod',
    'language', 'anchor_year_group',
]

model_df = df.drop(columns=leakage_and_id_cols, errors='ignore')

print("model_df shape:", model_df.shape)
model_df.columns

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Drop readmit_30d — that's the OTHER model's target, not a feature for LOS
model_df_los = model_df.drop(columns=['readmit_30d'], errors='ignore')

# Drop rows with unusable los (NaN or infinite)
bad_mask = model_df_los['los'].isna() | np.isinf(model_df_los['los'])
model_df_los = model_df_los[~bad_mask]

X = model_df_los.drop(columns=['los'])
y = model_df_los['los']

# Patient-level split so no patient appears in both train and test
groups = df.loc[model_df_los.index, 'subject_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Log-transform target — los is right-skewed
y_train_log = np.log(y_train)

xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train_log)

y_pred = np.clip(np.exp(xgb_model.predict(X_test)), 0, None)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"XGBoost — MAE: {mae:.2f} days | RMSE: {rmse:.2f} days | R²: {r2:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Reuses X_train, X_test, y_train, y_test, y_train_log from Cell 1 — same split, fair comparison

rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train_log)

y_pred_rf = np.clip(np.exp(rf_model.predict(X_test)), 0, None)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest — MAE: {mae_rf:.2f} days | RMSE: {rmse_rf:.2f} days | R²: {r2_rf:.4f}")

## Model Comparison: Original Notebook vs. New Notebook (demographics-only)

Same feature set (demographics only) across all three models below — a direct check that the new notebook's models reproduce the original results correctly.

| Model | Notebook | MAE (days) | RMSE (days) | R² | R² scale |
|---|---|---|---|---|---|
| `model` (log-target) | Original (`real_eda.ipynb`) | 2.36 | 5.36 | 0.195 | ln scale |
| XGBoost | New notebook | 2.37 | 5.36 | 0.0725 | raw scale |
| Random Forest | New notebook | 2.38 | 5.40 | 0.0586 | raw scale |

**What this shows:**
- **MAE and RMSE match almost exactly** between the original `model` and the new XGBoost (2.36 vs 2.37 days MAE, 5.36 vs 5.36 days RMSE) — confirms the new notebook's XGBoost is a faithful reproduction of the original demographics-only model.
- **R² values are NOT comparable across the two notebooks** — the original computed R² on the log scale, the new notebook computes it on the raw scale. See scale note above; don't read the R² gap as a real difference in model quality.
- **Random Forest performs slightly worse than XGBoost** on this same feature set (MAE 2.38 vs 2.37, RMSE 5.40 vs 5.36) — a small gap, consistent with XGBoost's usual edge over vanilla Random Forest, though not dramatic on demographics alone.
- **None of these three beat `model_v2`/`model_v3`** (MAE 2.10–2.12 days) — reinforces that adding vitals + labs is what actually moves the needle, not the choice of algorithm (XGBoost vs. Random Forest) on the same limited feature set.

In [ ]:
import pandas as pd
from pathlib import Path

MIMIC_ROOT = Path(r"E:\MedAgent\data\raw\physionet.org\files\mimiciv\3.1")
SUBFOLDERS = ["hosp", "icu"]

def get_csv_columns(root=MIMIC_ROOT, subfolders=SUBFOLDERS):
    table_columns = {}
    for sub in subfolders:
        folder = root / sub
        # match both "name.csv" and "name.csv.gz" naming conventions
        files = sorted(set(folder.glob("*.csv")) | set(folder.glob("*.csv.gz")))

        if not files:
            print(f"WARNING: no files found in {folder} — check the path/name.")
            continue

        for file in files:
            # strip both ".gz" and ".csv" suffixes to get the clean table name
            table_name = file.name
            for suffix in (".gz", ".csv"):
                if table_name.endswith(suffix):
                    table_name = table_name[: -len(suffix)]

            try:
                df_head = pd.read_csv(file, nrows=0, compression="gzip")
                table_columns[f"{sub}/{table_name}"] = list(df_head.columns)
            except Exception as e:
                print(f"Could not read {file.name}: {e}")

    return table_columns


def print_columns_report(table_columns):
    for table, cols in table_columns.items():
        print(f"\n=== {table} ({len(cols)} columns) ===")
        for c in cols:
            print(f"  - {c}")


if __name__ == "__main__":
    table_columns = get_csv_columns()
    print(f"\nFound {len(table_columns)} tables total.")
    print_columns_report(table_columns)

    rows = [
        {"table": table, "column": col}
        for table, cols in table_columns.items()
        for col in cols
    ]
    columns_df = pd.DataFrame(rows)
    columns_df.to_csv("mimic_all_columns.csv", index=False)
    print(f"\nSaved {len(columns_df)} rows to mimic_all_columns.csv")

In [ ]:
import pandas as pd
from pathlib import Path

MIMIC_ROOT = Path(r"E:\MedAgent\data\raw\physionet.org\files\mimiciv\3.1")
HOSP = MIMIC_ROOT / "hosp"
ICU = MIMIC_ROOT / "icu"

def load(table, subfolder):
    # Match either naming convention, same as the column-scan script
    candidates = list(subfolder.glob(f"{table}.csv")) + list(subfolder.glob(f"{table}.csv.gz"))
    if not candidates:
        raise FileNotFoundError(f"No file found for table '{table}' in {subfolder}")
    file = candidates[0]
    return pd.read_csv(file, compression="gzip")

# 1. Base cohort - one row per ICU stay, target = los
icustays = load("icustays", ICU)
icustays["intime"] = pd.to_datetime(icustays["intime"])
icustays["outtime"] = pd.to_datetime(icustays["outtime"])

df = icustays[[
    "subject_id", "hadm_id", "stay_id",
    "first_careunit", "last_careunit",
    "intime", "outtime", "los"
]].copy()

# 2. Patient demographics (age/sex only — race/language excluded, see note to Alex)
patients = load("patients", HOSP)
df = df.merge(patients[["subject_id", "gender", "anchor_age"]], on="subject_id", how="left")

# 3. Admission-level features
admissions = load("admissions", HOSP)
for col in ["admittime", "dischtime", "edregtime", "edouttime"]:
    admissions[col] = pd.to_datetime(admissions[col])

admissions["hosp_los_days"] = (
    (admissions["dischtime"] - admissions["admittime"]).dt.total_seconds() / 86400
)
admissions["came_through_ed"] = admissions["edregtime"].notna().astype(int)
admissions["ed_los_hours"] = (
    (admissions["edouttime"] - admissions["edregtime"]).dt.total_seconds() / 3600
)

adm_features = admissions[[
    "hadm_id", "admission_type", "admission_location", "discharge_location",
    "insurance", "marital_status", "hosp_los_days", "came_through_ed",
    "ed_los_hours", "hospital_expire_flag"
]]
df = df.merge(adm_features, on="hadm_id", how="left")

# 4. Diagnosis burden
diagnoses = load("diagnoses_icd", HOSP)
num_diagnoses = diagnoses.groupby("hadm_id").size().rename("num_diagnoses").reset_index()
df = df.merge(num_diagnoses, on="hadm_id", how="left")
df["num_diagnoses"] = df["num_diagnoses"].fillna(0)

# 5. Procedure burden
procedures = load("procedures_icd", HOSP)
num_procedures = procedures.groupby("hadm_id").size().rename("num_procedures").reset_index()
df = df.merge(num_procedures, on="hadm_id", how="left")
df["num_procedures"] = df["num_procedures"].fillna(0)

# 6. Ward transfer volume (proxy for clinical instability)
transfers = load("transfers", HOSP)
num_transfers = transfers.groupby("hadm_id").size().rename("num_transfers").reset_index()
df = df.merge(num_transfers, on="hadm_id", how="left")
df["num_transfers"] = df["num_transfers"].fillna(0)

# 7. Medication burden (polypharmacy proxy)
prescriptions = load("prescriptions", HOSP)
num_medications = (
    prescriptions.groupby("hadm_id")["drug"].nunique()
    .rename("num_distinct_meds").reset_index()
)
df = df.merge(num_medications, on="hadm_id", how="left")
df["num_distinct_meds"] = df["num_distinct_meds"].fillna(0)

# 8. Admitting service (medicine vs surgery vs other)
services = load("services", HOSP)
first_service = (
    services.sort_values("transfertime")
    .groupby("hadm_id")["curr_service"].first()
    .rename("admitting_service").reset_index()
)
df = df.merge(first_service, on="hadm_id", how="left")

print(df.shape)
df.head()

## Feature Engineering — ICU Length of Stay (LOS)

**Target:** `los` from `icustays.csv` — ICU length of stay in days, computed directly by MIMIC-IV from `intime`/`outtime`.

**Cohort:** one row per ICU stay (`stay_id`), pulled from `icustays`.

### Features included

| Feature | Source table | Derivation |
|---|---|---|
| `gender`, `anchor_age` | `patients` | Direct pull, joined on `subject_id` |
| `admission_type`, `admission_location`, `discharge_location`, `insurance`, `marital_status` | `admissions` | Direct pull, joined on `hadm_id` |
| `hosp_los_days` | `admissions` | `dischtime - admittime`, in days |
| `came_through_ed` | `admissions` | Flag: `edregtime` is not null |
| `ed_los_hours` | `admissions` | `edouttime - edregtime`, in hours |
| `hospital_expire_flag` | `admissions` | Direct pull — in-hospital mortality flag |
| `num_diagnoses` | `diagnoses_icd` | Count of ICD codes per `hadm_id` — proxy for case complexity |
| `num_procedures` | `procedures_icd` | Count of ICD procedure codes per `hadm_id` |
| `num_transfers` | `transfers` | Count of ward/unit transfer events per `hadm_id` — proxy for clinical instability |
| `num_distinct_meds` | `prescriptions` | Count of distinct drug names per `hadm_id` — polypharmacy proxy |
| `admitting_service` | `services` | First `curr_service` recorded per `hadm_id` (e.g. medicine, surgery) |

Vitals and labs (`mean_gcs_verbal`, etc.) carry over unchanged from the existing `df_encoded_v2`/`v3` pipeline and are joined onto this base on `stay_id`/`hadm_id`.

### Deliberately excluded (pending discussion)

`race` and `language` are present in `admissions` but are **not** included in this feature set. Flagging for discussion before we decide whether/how to use them — want to align on that before they're active features again.

### Not yet explored (candidates for next pass)

- Elixhauser/Charlson comorbidity index (weighted diagnosis severity, vs. raw count)
- First-24-hour vitals/labs specifically (vs. stay-long means) — may better reflect admission severity rather than leaking outcome-adjacent info
- `emar`/`pharmacy` administration timing (vs. just prescription counts)